# Hybrid embedding preparation
Select a GPU runtime. Upload `colab_embeddings_inputs.zip` in the next cell. This job creates hybrid embeddings at batch size 256.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile
from google.colab import files

uploaded = files.upload()
archive = Path('colab_embeddings_inputs.zip')
assert archive.name in uploaded, 'Upload colab_embeddings_inputs.zip'
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.infolist():
        target = (Path('/content') / member.filename).resolve()
        assert target.is_relative_to('/content/ER'), 'Unexpected archive path'
    bundle.extractall('/content')
ER_ROOT = Path('/content/ER')
os.chdir(ER_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'training.prepare_embeddings',
                '--setup-dir', 'data/track_setup', '--device', 'cuda',
                '--batch-size', '256'], env={**os.environ, 'PYTHONPATH': str(ER_ROOT / 'src')}, check=True)


Keep `data/track_setup/shared_minilm__embeddings.npz` with the prepared training inputs. If using temporary Colab storage, download it before closing the runtime and put it in the local `ER/data/track_setup` directory. The three-track preflight will validate it before training.


In [ ]:
from google.colab import files
files.download('data/track_setup/shared_minilm__embeddings.npz')
